In [1]:
import random
import time
import pandas as pd
import numpy as np
import torch
from torch import nn
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, PrefixTuningConfig, TaskType
import evaluate

SEED = 24

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [2]:
MODEL_CHECKPOINT = "google-bert/bert-base-uncased"

EPOCHS = 10
LR = 2e-5
BATCH_SIZE = 8
WEIGHT_DECAY = 0.01

In [3]:
# Загрузка датасета и токенизатора
dataset = load_dataset("dair-ai/emotion")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [4]:
# Токенизация
def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True)

dataset = dataset.map(tokenize_batch, batched=True)

In [ ]:
NUM_LABELS = dataset['train'].features['label'].num_classes

In [6]:
data_collator = DataCollatorWithPadding(tokenizer)

In [7]:
# Метрики
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels, average='macro', zero_division=0)['precision'],
        'recall': recall.compute(predictions=preds, references=labels, average='macro', zero_division=0)['recall'],
        'f1_macro': f1.compute(predictions=preds, references=labels, average='macro')['f1'],
    }

In [ ]:
# Универсальная функция для запуска эксперимента
# Возвращает словарь с результатами: метрики, время, ресурсы
def run_experiment(model, name):
    args = TrainingArguments(
        output_dir=f"./results/{name}",
        eval_strategy="epoch",
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        seed=SEED,
        load_best_model_at_end=False,
        logging_steps=50,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # 2) Считаем обучаемые параметры
    num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 3) Тренируем и сбрасываем статистики память/таймер
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    memory = torch.cuda.memory_allocated() / (1024 ** 2)

    # 4) Оценка на тесте
    eval_res = trainer.evaluate(dataset["test"])

    # 5) Печать результатов
    print(f"\n[{name}]")
    print(f"  Params:       {num_trainable:,}")
    print(f"  Time (s):     {elapsed:.1f}")
    print(f"  GPU memory (MB):{memory:.1f}")
    for metric in ("eval_accuracy", "eval_precision", "eval_recall", "eval_f1_macro"):
        print(f"  {metric[5:]:<12}: {eval_res[metric]:.4f}")

    return {
        "method": name,
        "params": num_trainable,
        "time_s": elapsed,
        "mem_mb": memory,
        **{k: v for k, v in eval_res.items() if k.startswith("eval_")},
    }

# 1. Базовые метрики без дообучения

In [9]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=NUM_LABELS)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
args = TrainingArguments(
    output_dir='./results/base', 
    per_device_eval_batch_size=BATCH_SIZE)

base_trainer = Trainer(
    model=base_model,
    args=args,
    eval_dataset=dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [11]:
base_metrics = base_trainer.evaluate(dataset['test'])
base_metrics

{'eval_loss': 1.7509193420410156,
 'eval_model_preparation_time': 0.0,
 'eval_accuracy': 0.2235,
 'eval_precision': 0.11783263622878191,
 'eval_recall': 0.16420509797910912,
 'eval_f1_macro': 0.11645992936366267,
 'eval_runtime': 2.3071,
 'eval_samples_per_second': 866.901,
 'eval_steps_per_second': 108.363}

# 2. Full Fine-Tuning

In [ ]:
# Список для хранения всех результатов
results = []
#полностью дообучаем BERT на задаче классификации
full_model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=NUM_LABELS)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
results.append(run_experiment(full_model, 'full_finetuning'))

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,0.186200,0.184487,0.936000,0.916070,0.906187,0.909073
2,0.153400,0.224524,0.936500,0.924292,0.901456,0.912143
3,0.130900,0.198975,0.934500,0.903651,0.918700,0.909626
4,0.128700,0.282160,0.933000,0.904577,0.909662,0.906692
5,0.039800,0.321492,0.932500,0.914314,0.906235,0.908312
6,0.042700,0.342846,0.937500,0.907731,0.923291,0.914103
7,0.028900,0.424865,0.936500,0.914337,0.910369,0.912000
8,0.020100,0.459898,0.936500,0.910206,0.916317,0.913051
9,0.016100,0.416134,0.939000,0.922427,0.907591,0.914568
10,0.039200,0.461538,0.937000,0.911109,0.911672,0.911365



[full_finetuning]
  Params:       109,486,854
  Time (s):     1819.7
  GPU memory (MB):1691.8
  accuracy    : 0.9315
  precision   : 0.8917
  recall      : 0.8895
  f1_macro    : 0.8904


# 3. Linear Probing с кастомной головой

Создаём кастомную классификационную голову с несколькими слоями для лучшей нелинейности, Dropout служит для регуляризации, чтобы избежать переобучения, LayerNorm стабилизирует обучение
- Архитектура головы: LayerNorm -> Linear(hidden_size->hidden_size/2) -> ReLU -> Dropout(0.1) -> Linear(hidden_size/2->hidden_size/4) -> ReLU -> Dropout(0.1) -> Linear(hidden_size/4->num_labels)

In [ ]:
class CustomProbe(nn.Module):
    def __init__(self, hidden_size, num_labels):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 4, num_labels)
        )
    def forward(self, x):
        return self.head(x)


In [15]:
probe_model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=NUM_LABELS)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
# Заморозка всех слоев, кроме головы
for name, param in probe_model.named_parameters():
    param.requires_grad = name.startswith('classifier')

In [17]:
# Замена головы
probe_model.classifier = CustomProbe(probe_model.config.hidden_size, NUM_LABELS)

In [18]:
results.append(run_experiment(probe_model, 'linear_probing'))

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,1.591300,1.554817,0.380500,0.127512,0.189905,0.140736
2,1.506800,1.520913,0.423000,0.143648,0.213873,0.162874
3,1.489000,1.487996,0.445000,0.148625,0.227206,0.174518
4,1.443100,1.459423,0.473500,0.156544,0.249583,0.192248
5,1.429300,1.433101,0.478500,0.157044,0.249830,0.192662
6,1.425700,1.416267,0.478000,0.156898,0.249129,0.192195
7,1.467200,1.408368,0.480500,0.241335,0.249754,0.193580
8,1.496300,1.402054,0.489000,0.161143,0.256922,0.198053
9,1.443100,1.397023,0.489500,0.161227,0.257027,0.198155
10,1.369500,1.396441,0.490500,0.162142,0.258428,0.199133



[linear_probing]
  Params:       371,910
  Time (s):     481.7
  GPU memory (MB):1280.6
  accuracy    : 0.4850
  precision   : 0.1606
  recall      : 0.2514
  f1_macro    : 0.1960


# 4. Prefix Tuning

Выбор Prefix Tuning:
- В отличие от prompt tuning, добавляет обучаемые префиксы не в текст, а во внутренние проекционные матрицы
- Глубже влияет на слои attention, что важно для тонкой классификации эмоций
- чаще работает лучше на средних и малых моделях для classification

In [ ]:
prefix_cfg = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    num_virtual_tokens=20,
    prefix_projection=True,
)

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=NUM_LABELS)
pt_model = get_peft_model(model, prefix_cfg)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
results.append(run_experiment(pt_model, 'prefix_tuning'))

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,1.294400,1.205464,0.589000,0.201068,0.313883,0.241972
2,0.953200,0.915514,0.741000,0.508815,0.514185,0.500138
3,0.773400,0.712415,0.800000,0.710355,0.593435,0.569963
4,0.645600,0.565802,0.880500,0.890758,0.768739,0.785583
5,0.590900,0.513057,0.901500,0.900444,0.833499,0.853282
6,0.575300,0.470415,0.916000,0.893849,0.868313,0.879530
7,0.589500,0.452790,0.923000,0.904848,0.882984,0.892350
8,0.579400,0.435854,0.923000,0.900872,0.887280,0.893284
9,0.533200,0.430533,0.920500,0.901352,0.883909,0.891505
10,0.447100,0.428912,0.922500,0.897837,0.892830,0.895057



[prefix_tuning]
  Params:       14,780,160
  Time (s):     636.7
  GPU memory (MB):1863.9
  accuracy    : 0.9215
  precision   : 0.8780
  recall      : 0.8763
  f1_macro    : 0.8770


# 5. LoRA

In [ ]:
# перебираем разные r
r_values = [4, 8, 16, 32]
for r in r_values:
    # фиксируем коэффициент масштабирования lora_alpha равным рангу r
    # чтобы сравнить влияние только ранга без дополнительного масштабирования
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=r,
        lora_alpha=r,  # alpha=r для упрощённого масштабирования адаптера
        lora_dropout=0.1,
        target_modules=["query", "key", "value"],
    )
    # Создание и дообучение модели с текущим ранком
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=NUM_LABELS).to(DEVICE)
    model_r = get_peft_model(model, lora_cfg)
    results.append(run_experiment(model_r, f'lora_r{r}'))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,1.510900,1.475986,0.456000,0.149252,0.234735,0.180487
2,1.170800,1.204695,0.558500,0.190729,0.297386,0.229452
3,1.134500,1.112095,0.578500,0.197597,0.307850,0.237669
4,1.054900,1.067145,0.587000,0.202117,0.312869,0.241770
5,0.984400,1.010094,0.599500,0.422911,0.331736,0.279431
6,0.928900,0.948983,0.643000,0.508531,0.396591,0.378530
7,0.985000,0.899424,0.669000,0.647199,0.442132,0.432758
8,1.026000,0.862187,0.679000,0.671574,0.463743,0.454676
9,0.949700,0.840745,0.691000,0.691453,0.485247,0.482940
10,0.788800,0.834237,0.696500,0.712466,0.495104,0.496797


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[lora_r4]
  Params:       225,798
  Time (s):     667.9
  GPU memory (MB):2173.2
  accuracy    : 0.7285
  precision   : 0.7273
  recall      : 0.5142
  f1_macro    : 0.5214


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,1.271800,1.195100,0.564500,0.193077,0.300956,0.232036
2,0.964800,1.006327,0.616500,0.465264,0.368910,0.324973
3,0.872300,0.834767,0.686500,0.687176,0.489024,0.470994
4,0.748700,0.751672,0.729000,0.720572,0.550869,0.556776
5,0.660900,0.662695,0.758000,0.761106,0.605327,0.623884
6,0.598100,0.597517,0.779500,0.752826,0.652836,0.672006
7,0.615300,0.547351,0.806000,0.795445,0.683540,0.714460
8,0.707500,0.521202,0.821500,0.804226,0.711672,0.743119
9,0.600400,0.500954,0.830500,0.811650,0.728672,0.757106
10,0.453000,0.497441,0.834000,0.814909,0.732230,0.760631



[lora_r16]
  Params:       889,350
  Time (s):     678.4
  GPU memory (MB):2180.1
  accuracy    : 0.8460
  precision   : 0.8126
  recall      : 0.7359
  f1_macro    : 0.7634


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,1.182600,1.109221,0.581000,0.200521,0.309763,0.239458
2,0.748800,0.837924,0.686500,0.686704,0.488573,0.466989
3,0.695200,0.671996,0.758000,0.738410,0.591183,0.613043
4,0.586000,0.556220,0.806500,0.780537,0.691253,0.715750
5,0.468700,0.472981,0.839000,0.813335,0.749128,0.769427
6,0.441500,0.431623,0.857000,0.829967,0.793166,0.806947
7,0.445600,0.401596,0.871500,0.844813,0.797895,0.813961
8,0.505500,0.379720,0.882500,0.858443,0.828433,0.841271
9,0.409400,0.370724,0.882500,0.857225,0.830048,0.841859
10,0.330800,0.367149,0.886000,0.860372,0.834288,0.845671



[lora_r32]
  Params:       1,774,086
  Time (s):     681.8
  GPU memory (MB):2190.2
  accuracy    : 0.8900
  precision   : 0.8526
  recall      : 0.8277
  f1_macro    : 0.8393


При значительном росте метрик время дообучения, необходимая память примерно не меняется, поэтому наилучшим образом показала себя lora_r32

# 6. Сводка результатов

In [23]:
summary = pd.DataFrame(results)[[
    'method',
    'params',
    'time_s',
    'mem_mb',
    'eval_accuracy',
    'eval_precision',
    'eval_recall',
    'eval_f1_macro',
]]
summary

,method,params,time_s,mem_mb,eval_accuracy,eval_precision,eval_recall,eval_f1_macro
0,full_finetuning,109486854,1819.676100,1691.849609,0.9315,0.891669,0.889489,0.890367
1,linear_probing,371910,481.746375,1280.607910,0.4850,0.160587,0.251435,0.195967
2,prefix_tuning,14780160,636.672108,1863.912109,0.9215,0.878028,0.876333,0.876969
3,lora_r4,225798,667.937377,2173.231445,0.7285,0.727251,0.514190,0.521377
4,lora_r8,446982,680.476384,2175.012695,0.7985,0.778082,0.649887,0.680498
5,lora_r16,889350,678.380420,2180.075195,0.8460,0.812645,0.735902,0.763399
6,lora_r32,1774086,681.776644,2190.200195,0.8900,0.852618,0.827700,0.839294


- full_finetuning даёт наилучшее качество(eval_accuracy = 0.9315), но требует максимальных ресурсов, обучающих параметров и времени
- linear_probing очень экономен по параметрам и памяти, но качество существенно ниже (accuracy = 0.4850)
- prefix_tuning близок к full_finetuning по качеству, при этом экономия времени и параметров, однако по памяти тяжелее
- LoRA с разными рангами примерно похожи по времени и памяти, но с r=32 достигает 0.89 accuracy, балансируя между временем и параметрами
- По сложности: линейная linear_probing < prefix_tuning/lora < full-tuning